# Fine-tuned Model Inference (Google Colab)

Load **base model + PEFT adapter** (`outputs/smollm2-bet-advisor`) and run inference on eval/test questions.

## Setup: Google Drive Structure

### 1. Folder structure

Create a project folder in your Google Drive root (e.g. `genai_hw3`):

```
My Drive/
└── genai_hw3/              ← Project root (customize name if needed)
    ├── data/                ← Input data
    │   ├── eval.jsonl       ← Upload eval.jsonl here
    │   └── test.jsonl       ← Upload test.jsonl here
    └── outputs/             ← From training (train_lora_colab.ipynb)
        └── smollm2-bet-advisor/   ← PEFT adapter (required)
```

### 2. Required files

- **Adapter**: `outputs/smollm2-bet-advisor/` — must exist from training first
- **Input**: `data/eval.jsonl` and/or `data/test.jsonl` — at least one required

### 3. Output

Results are written to `outputs/finetuned_outputs.jsonl`.

## 0. Install Dependencies & Enable GPU

In [ ]:
# Colab menu: Runtime → Change runtime type → Hardware accelerator: GPU
!pip install -q transformers peft

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None (use CPU)'}")

## 1. Mount Google Drive & Path Configuration

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# ========== Modify if using a different folder name ==========
PROJECT_FOLDER = "genai_hw3"

project_root = Path("/content/drive/MyDrive") / PROJECT_FOLDER
data_dir = project_root / "data"
output_dir = project_root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

eval_path = data_dir / "eval.jsonl"
test_path = data_dir / "test.jsonl"
adapter_path = output_dir / "smollm2-bet-advisor"
output_path = output_dir / "finetuned_outputs.jsonl"

print(f"Project root: {project_root}")
print(f"Adapter: {adapter_path} (exists: {adapter_path.exists()})")
print(f"Eval: {eval_path} (exists: {eval_path.exists()})")
print(f"Test: {test_path} (exists: {test_path.exists()})")
print(f"Output: {output_path}")

if not adapter_path.exists():
    raise FileNotFoundError(
        f"Adapter not found at {adapter_path}. Run train_lora_colab.ipynb first."
    )

if not eval_path.exists() and not test_path.exists():
    raise FileNotFoundError(
        f"No eval.jsonl or test.jsonl found in {data_dir}. Upload at least one."
    )

## 2. Load Base Model + PEFT Adapter

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print(f"Loading base model: {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

print(f"Loading PEFT adapter: {adapter_path}")
model = PeftModel.from_pretrained(model, str(adapter_path))

if not torch.cuda.is_available():
    model = model.to("cpu")

print(f"Model loaded. Device: {next(model.parameters()).device}")

## 3. Load Evaluation Data

In [ ]:
import json

def load_jsonl(path: Path) -> list[dict]:
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                items.append(json.loads(line))
    return items

eval_data = load_jsonl(eval_path) if eval_path.exists() else []
test_data = load_jsonl(test_path) if test_path.exists() else []
questions = eval_data + test_data

NUM_QUESTIONS = None  # None = all, or set e.g. 15 for quick test
if NUM_QUESTIONS is not None:
    questions = questions[:NUM_QUESTIONS]
    print(f"(Limited to first {NUM_QUESTIONS} questions)")

print(f"Eval: {len(eval_data)}, Test: {len(test_data)}, Total: {len(questions)}")

## 4. Build Prompt (Same as base_inference)

In [ ]:
SYSTEM_PROMPT = """You are an expert NBA player data analyst. Your task is to analyze betting questions (over/under) using historical player statistics.

## Tree of Thought Reasoning Framework

Build your reasoning as a tree with these main branches (evaluate each, then synthesize):

1. **Sample & Statistics Branch**: For each context filter (all games, last N games, starter vs bench, with/without star teammates), assess:
   - n_games: Is sample size sufficient? (n < 10 → low weight)
   - p_over, p_under, mean, std: Which context favors over vs under?
   - Conflict between contexts → note uncertainty

2. **Lineup/Teammate Branch**: How does starter vs bench, or with/without star teammates, change the stats? Does lineup context support or contradict the main trend?

3. **Risk & Synthesis Branch**: Given the above, what is the net signal? If branches conflict or sample is weak → prefer "avoid".

## Output Format (JSON only, no markdown)

{
  "decision": "over" | "under" | "avoid",
  "confidence": 0.0 to 1.0,
  "reasoning": {
    "tree_of_thought": [
      {"step": 1, "branch": "sample_stats", "thought": "...", "conclusion": "..."},
      {"step": 2, "branch": "lineup_teammate", "thought": "...", "conclusion": "..."},
      {"step": 3, "branch": "synthesis", "thought": "...", "conclusion": "..."}
    ]
  },
  "summary": "One-sentence conclusion"
}

Each step must have: branch (which dimension), thought (your analysis), conclusion (what this branch implies for the decision).
Respond with ONLY valid JSON, no markdown or extra text."""

def build_prompt(instruction: str, input_str: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{instruction}\n\nStatistics:\n{input_str}"},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

## 5. Inference Loop

In [ ]:
def generate(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=4096,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print("Starting inference...")
results = []
for i, item in enumerate(questions):
    prompt = build_prompt(item["instruction"], item["input"])
    model_output = generate(prompt)
    results.append({
        "question": item["instruction"],
        "input": item["input"],
        "model_output": model_output.strip(),
        "ground_truth": item["output"],
    })
    if (i + 1) % 10 == 0:
        print(f"  Processed {i + 1}/{len(questions)} questions")

print(f"Completed. Total: {len(results)} questions")

## 6. Write Results to Google Drive

In [ ]:
with open(output_path, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Written to: {output_path}")
print(f"Total: {len(results)} items")
if results:
    print("\nExample (first 300 chars of first model_output):")
    print(results[0]["model_output"][:300])